In this notebook we will make patients sequencies similar to those described in BEHRT paper


In [ ]:
# load the data
# we are going to use data from PHE_2024.db
import pandas as pd
path = '/Users/annagerasimenko/phe_2024_updated_scripts_and_models/phe_aggregated_2024.csv'
# load the data
df = pd.read_csv(path)

# we are interested in the following columns:
df.head()

In [ ]:
print(f'The number of unqiue patients is {len(set(df["enrolid"])):,}')



In [ ]:
path2 = '/Users/annagerasimenko/phecodeX_info.csv'  
phecode_info = pd.read_csv(path2)


In [ ]:
phecode_info.head()

In [ ]:
phecode_to_phenotype = phecode_info.set_index('phecode')['phecode_string']
df['phenotype'] = df['phecode'].map(phecode_to_phenotype)
df.head()

In [ ]:
print(f"shape before dropping na: {df.shape}")
df = df.dropna(subset=['phenotype'])
print(f"shape after dropping na: {df.shape}")


In [ ]:
# df['phenotype'].isna().sum()

In [ ]:
# df.loc[df['phenotype'].isna(), 'phecode'].unique()

In [ ]:
# df['phenotype'].value_counts()

In [ ]:
import sqlite3

demo_con = sqlite3.connect('/Volumes/FastSSD_2/DEMO_2024.db')

print("Loading full DEMO table...")
demo_df = pd.read_sql("SELECT enrolid, dobyr FROM DEMO", demo_con)
demo_con.close()
print(f"Loaded {len(demo_df):,} rows from DEMO")


In [ ]:
demo_df = demo_df.drop_duplicates(subset='enrolid')
print(f"Unique patients in DEMO: {len(demo_df):,}")

patient_ids = set(df['enrolid'].unique())
demo_df = demo_df[demo_df['enrolid'].isin(patient_ids)]
print(f"Matched to our dataset: {len(demo_df):,} patients")
demo_df.head()

In [ ]:
df = df.merge(demo_df, on='enrolid', how='left')
print(f"Rows with missing dobyr: {df['dobyr'].isna().sum():,}")
df.head()

In [ ]:
save_path = '/Users/annagerasimenko/embeddings_BEHRT/df_with_dobyr.pkl'
df.to_pickle(save_path)
print(f"Saved df to {save_path}")
print(f"Shape: {df.shape}")

# To load later:
# df = pd.read_pickle('/Users/annagerasimenko/embeddings_BEHRT/df_with_dobyr.pkl')

## Reviewing the outcomes of building_patients_seqs.py

In [ ]:
import pickle
with open('/Users/annagerasimenko/embeddings_BEHRT/age2idx.pkl', 'rb') as f:
    age2idx = pickle.load(f)
print(type(list(age2idx.keys())[0]))  # check if keys are int or str
print(dict(list(age2idx.items())[:10]))  # see first 10 entries

In [ ]:
import pandas as pd

sequences = pd.read_pickle('/Users/annagerasimenko/embeddings_BEHRT/patient_sequences.pkl')

patient = sequences.iloc[0]
print(f"Patient: {patient['enrolid']}")
print(f"Sequence length: {len(patient['code'])} tokens")
print(f"Number of visits: {patient['code'].count('SEP')}")
print()

visit_num = 1
print(f"--- Visit {visit_num} ---")
for code, age in zip(patient['code'], patient['age']):
    if code == 'SEP':
        visit_num += 1
        print(f"--- Visit {visit_num} ---")
    else:
        print(f"  {code}  (age {age})")

In [ ]:
# Find a patient who has age 1 in their sequence
for i, row in sequences.iterrows():
    if 1 in row['age']:
        print(f"Patient: {row['enrolid']}, sequence length: {len(row['code'])}")
        visit_num = 1
        print(f"--- Visit {visit_num} ---")
        for code, age in zip(row['code'], row['age']):
            if code == 'SEP':
                visit_num += 1
                print(f"--- Visit {visit_num} ---")
            else:
                print(f"  {code}  (age {age})")
        break